# Predictive Anayltics: Support Vector Machines

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [1]:
from run_config import PATHS

In [2]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

## Preparations

In [3]:
INPUT = PATHS.train_test_dir

In [4]:
#GRID_SAMPLE = 35_000 # full or number
SPATIAL_UNIT = "COMMUNITY_AREAS" # HEXAGON
SPATIAL_ENCODING = "latlong" # options: embedding, latlong
MODE = "full" # options: full, sample
TIME_UNIT = "24H" # options: 1H, 4H, 24H
H3_RES = "7" # options 7,8

In [5]:
# "Settings" / Decisions for the training data

DATA_PATH_TRAIN = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TRAIN.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TEST.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_VAL.parquet"
if SPATIAL_UNIT == "HEXAGON":
    DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
    DATA_PATH_TRAIN = INPUT / F"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
    DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"

MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_demand",
    "date",
]

Load data and select features and target

In [6]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
val = pl.scan_parquet(DATA_PATH_VAL)
test = pl.scan_parquet(DATA_PATH_TEST)

In [7]:
train_df = train.collect()
val_df = val.collect()
test_df = test.collect()

train_df = train_df.to_pandas()
val_df = val_df.to_pandas()
test_df = test_df.to_pandas()

In [8]:
train_df.head()
type(train_df)

pandas.DataFrame

In [9]:
# prepare data
# calculate median to split in low/high demand
# when trip_count above 50 percent use "high", when below or equal to 50 percent low
train_median = train_df["trip_count"].median()
train_df["trip_demand"] = np.where(train_df["trip_count"] > train_median, "high", "low")

val_median = val_df["trip_count"].median()
val_df["trip_demand"] = np.where(val_df["trip_count"] > val_median, "high", "low")

test_median = test_df["trip_count"].median()
test_df["trip_demand"] = np.where(test_df["trip_count"] > test_median, "high", "low")

In [10]:
train_df = train_df.sample(n=20000, random_state=42)
train_df_grid = train_df.sample(n=5000, random_state=42)

In [11]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

# Community_area is a categorical id, not a numeric quantity, so one-hot encode it
X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])

# Keep the dummy columns before scaling turns X_train into a plain array
train_columns = X_train.columns

# Make sure val/test have the same dummy columns as train (in case a community_area is missing)
X_val = X_val.reindex(columns=train_columns, fill_value=0)
X_test = X_test.reindex(columns=train_columns, fill_value=0)

y_train = train_df[TARGET_COL]
y_val = val_df[TARGET_COL]
y_test = test_df[TARGET_COL]

# SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Features:", feature_cols)
print("Target:", y_train.dtypes)

Features: ['month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'hour_sin', 'hour_cos', 'is_holiday', 'community_area', 'weather_station_distance_km', 'food_drink', 'landmark', 'shop', 'train_station', 'tmpc', 'relh', 'sknt', 'p01m', 'vsby', 'wind_dir_sin', 'wind_dir_cos', 'station_observed', 'weather_imputed', 'precipitation_missing', 'weather_rain', 'weather_snow', 'weather_fog_mist', 'weather_thunder', 'weather_freezing', 'precipitation_trace', 'weather_qc_corrected', 'skyc1_CLR', 'skyc1_FEW', 'skyc1_SCT', 'skyc1_BKN', 'skyc1_OVC', 'skyc1_VV', 'weather_station_MDW', 'weather_station_ORD', 'weather_station_IGQ']
Target: uint32


In [12]:
# Create X and y for grid search (same encoding + scaling as the full training set)
X_train_grid = pd.get_dummies(train_df_grid[feature_cols], columns=["community_area"])
X_train_grid = X_train_grid.reindex(columns=train_columns, fill_value=0)
X_train_grid = scaler.transform(X_train_grid)

y_train_grid = train_df_grid[TARGET_COL]

In [13]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,trip_demand
21536,2025-09-23,9,2,0,-0.866025,-5.000000e-01,0.781831,0.623490,0.0,1.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00,No trips,low
5516,2025-05-26,5,1,0,0.866025,-5.000000e-01,0.000000,1.000000,0.0,1.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00,No trips,low
18731,2026-01-05,1,1,0,0.000000,1.000000e+00,0.000000,1.000000,0.0,1.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00,No trips,low
21937,2025-05-23,5,5,0,0.866025,-5.000000e-01,-0.433884,-0.900969,0.0,1.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00,No trips,low
11140,2025-05-08,5,4,0,0.866025,-5.000000e-01,0.433884,-0.900969,0.0,1.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00,No trips,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8531,2025-04-18,4,5,0,1.000000,6.123234e-17,-0.433884,-0.900969,0.0,1.0,...,1044.5,0.488998,0.0,55.0,29943.3,14.018399,3.5,98.25,Mobile,high
24120,2025-01-21,1,2,0,0.000000,1.000000e+00,0.781831,0.623490,0.0,1.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00,No trips,low
9095,2025-08-10,8,7,0,-0.500000,-8.660254e-01,-0.781831,0.623490,0.0,1.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00,No trips,low
23583,2025-09-08,9,1,0,-0.866025,-5.000000e-01,0.000000,1.000000,0.0,1.0,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.00,No trips,low


In [14]:
model = SVR()

In [15]:
param_grid_linear = {
    "C": [0.1, 1, 10],
    "epsilon": [0.01, 0.1, 0.5, 1],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [0.1, 1, 10],
    "epsilon": [0.01, 0.1, 0.5, 1],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "epsilon": [0.01, 0.1, 0.5, 1],
    "kernel": ["poly"],
    "degree": [3, 4, 5],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVR(),
        param_grid=grid,
        cv=3,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_train_grid, y_train_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

KeyboardInterrupt: 

In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 0.1, 'degree': 4, 'epsilon': 1, 'gamma': 0.1, 'kernel': 'poly'}
Best CV score: 0.6518731893734491


In [ ]:
# Train SVR with the best hyperparameters found by grid search
best_params = grid_search.best_params_
model = SVR(**best_params)
model.fit(X_train, y_train)

: 

: 

In [ ]:
# Make prediction 
y_pred = model.predict(X_test)

In [ ]:
y_pred

array([-1.31339049,  0.81902821,  1.97237673, ..., -0.64759251,
       -2.55128149, -0.49586741], shape=(223531,))

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 5.5602808155517005
MSE: 298.15442265439026
RMSE: 17.267148654435978
R2 Score: 0.7588652846925792
